# vLLM Embedding Benchmark

Use this notebook on an A100 40 GB MIG slice to compare embedding models without paying for the full heavyweight notebook path.

The default candidate list below is intentionally limited to models that are explicitly shown or directly exemplified in the current vLLM pooling-model docs, so the first sweep stays close to documented support.

Main slowdown causes in `mimic_umls_pipeline.ipynb`:
- `MIMIC_DISCHARGE_LIMIT="all"` causes full-note extraction and graph builds.
- Optional UMLS normalization adds a large number of HTTP lookups.
- Every generation-model x embedding-model combination triggers a separate full index build.
- HTML and JPEG graph rendering run inside the same loop.
- `vllm_offline` loads models in-process, so model swaps are expensive.

This notebook keeps the experiment narrow: one generation model, a small embedding sweep, small note subset, small eval sample, and no graph rendering.

In [ ]:
from __future__ import annotations

import asyncio
import sys
import time
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))
load_dotenv(repo_root / ".env")

from eval.embedding_benchmark import EmbeddingBenchmarkConfig, run_embedding_benchmark

In [ ]:
# A100 40 GB MIG-friendly defaults.
GENERATION_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODELS = [
    "Qwen/Qwen3-Embedding-0.6B",
    "google/embeddinggemma-300m",
    "BAAI/bge-base-en-v1.5",
    "Snowflake/snowflake-arctic-embed-m-v1.5",
]
PROVIDER = "vllm"
USE_UMLS = False
SCHEMA_GUIDED = False
SAMPLE_SIZE = 5
NOTE_LIMIT = 25
NOTE_MAX_CHARS = 2000
NOTE_TYPE = "DS"
INPUT_DIR = repo_root / "data" / "evidence" / "mimic_discharge_subset"
MIMIC_CSV = repo_root / "data" / "mimic_iv_note" / "discharge.csv"
OUTPUT_ROOT = repo_root / "output" / "embedding_benchmark" / time.strftime("%Y%m%dT%H%M%S")

print("generation_model:", GENERATION_MODEL)
print("embedding_models:", EMBEDDING_MODELS)
print("provider:", PROVIDER)
print("output_root:", OUTPUT_ROOT)
print("note: if you add jinaai/jina-embeddings-v3 later, serve it with trust_remote_code as documented by vLLM.")

In [ ]:
results = asyncio.run(
    run_embedding_benchmark(
        EmbeddingBenchmarkConfig(
            input_dir=INPUT_DIR,
            output_root=OUTPUT_ROOT,
            generation_model=GENERATION_MODEL,
            embedding_models=tuple(EMBEDDING_MODELS),
            provider=PROVIDER,
            use_umls=USE_UMLS,
            schema_guided=SCHEMA_GUIDED,
            note_limit=NOTE_LIMIT,
            note_max_chars=NOTE_MAX_CHARS,
            note_type=NOTE_TYPE,
            mimic_csv=MIMIC_CSV,
            sample_size=SAMPLE_SIZE,
        )
    )
)

results_df = pd.DataFrame([asdict(result) for result in results])
results_df.sort_values(["exact_match", "mean_query_seconds"], ascending=[False, True], na_position="last")

In [ ]:
if not results_df.empty:
    ranked = results_df.copy()
    ranked["speed_rank"] = ranked["mean_query_seconds"].rank(method="min")
    if ranked["exact_match"].notna().any():
        ranked["quality_rank"] = ranked["exact_match"].rank(method="min", ascending=False)
        ranked["combined_rank"] = ranked["speed_rank"] + ranked["quality_rank"]
        display(ranked.sort_values(["combined_rank", "quality_rank", "speed_rank"]))
    else:
        display(ranked.sort_values(["speed_rank"]))
else:
    print("No benchmark rows were produced.")